In [ ]:
# =============================================================
#   BANK QUEUE MANAGEMENT SYSTEM
#   ระบบจัดการบัตรคิวธนาคาร
#   โครงสร้างข้อมูล: Queue (LinkedList), Stack, Heap, Array
#   Algorithms: Quick Sort, Merge Sort, Heap Sort, Insertion Sort,
#               Shell Sort, Radix Sort, Bucket Sort, Tim Sort
# =============================================================

import time
import heapq
from datetime import datetime
import json # Import json for saving/loading state
import atexit # Import atexit for automatic saving


# ──────────────────────────────────────────────
# DATA STRUCTURE: Node (LinkedList)
# ──────────────────────────────────────────────
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None


# ──────────────────────────────────────────────
# DATA STRUCTURE: Queue (LinkedList-based)
# ──────────────────────────────────────────────
class Queue:
    def __init__(self):
        self.head = self.tail = None
        self.size = 0

    def enqueue(self, data):
        node = Node(data)
        if self.tail:
            self.tail.next = node
        self.tail = node
        if not self.head:
            self.head = node
        self.size += 1

    def dequeue(self):
        if not self.head:
            return None
        data = self.head.data
        self.head = self.head.next
        if not self.head:
            self.tail = None
        self.size -= 1
        return data

    def peek(self):
        return self.head.data if self.head else None

    def to_list(self):
        result, cur = [], self.head
        while cur:
            result.append(cur.data)
            cur = cur.next
        return result

    def is_empty(self):
        return self.size == 0


# ──────────────────────────────────────────────
# DATA STRUCTURE: Stack (History)
# ──────────────────────────────────────────────
class Stack:
    def __init__(self):
        self._data = []

    def push(self, item):
        self._data.append(item)

    def pop(self):
        return self._data.pop() if self._data else None

    def peek(self):
        return self._data[-1] if self._data else None

    def to_list(self):
        return list(reversed(self._data)) # To show latest first

    def from_list(self, data_list):
        self._data = list(reversed(data_list)) # To restore internal order

    def is_empty(self):
        return len(self._data) == 0


# ──────────────────────────────────────────────
# MODEL: QueueTicket
# ──────────────────────────────────────────────
STATUS = {
    "WAITING":   "⏳ รอเรียก",
    "SERVING":   "✅ กำลังให้บริการ",
    "DONE":      "🏁 เสร็จสิ้น",
    "CANCELLED": "❌ ยกเลิก",
}

SERVICES = {
    "A": ("ฝาก/ถอนเงิน",        "A"),
    "B": ("เปิดบัญชีใหม่",       "B"),
    "C": ("สินเชื่อ",            "C"),
    "D": ("บัตรเครดิต",          "D"),
    "E": ("แลกเปลี่ยนเงินตรา",  "E"),
}

# Moved _ticket_counter to BankQueueSystem instance
def new_ticket(service_code: str, _ticket_counter: dict) -> dict:
    prefix = SERVICES[service_code][1]
    _ticket_counter[prefix] = _ticket_counter.get(prefix, 0) + 1
    return {
        "id":          f"{prefix}{_ticket_counter[prefix]:03d}",
        "service":     SERVICES[service_code][0],
        "prefix":      prefix,
        "number":      _ticket_counter[prefix],
        "status":      "WAITING",
        "issued_at":   datetime.now().strftime("%H:%M:%S"),
        "served_at":   None,
        "done_at":     None,
        "wait_secs":   0,
        "serve_secs":  0,
    }


# ──────────────────────────────────────────────
# SORTING ALGORITHMS
# ──────────────────────────────────────────────

# 1. Quick Sort
def quick_sort(arr, key=lambda x: x):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left   = [x for x in arr if key(x) < key(pivot)]
    middle = [x for x in arr if key(x) == key(pivot)]
    right  = [x for x in arr if key(x) > key(pivot)]
    return quick_sort(left, key) + middle + quick_sort(right, key)


# 2. Merge Sort
def merge_sort(arr, key=lambda x: x):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort(arr[:mid], key)
    right = merge_sort(arr[mid:], key)
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]


# 3. Heap Sort
def heap_sort(arr, key=lambda x: x):
    arr = list(arr)
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):
        _heapify(arr, n, i, key)
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        _heapify(arr, i, 0, key)
    return arr


def _heapify(arr, n, i, key):
    largest, l, r = i, 2*i+1, 2*i+2
    if l < n and key(arr[l]) > key(arr[largest]):
        largest = l
    if r < n and key(arr[r]) > key(arr[largest]):
        largest = r
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        _heapify(arr, n, largest, key)


# 4. Insertion Sort
def insertion_sort(arr, key=lambda x: x):
    arr = list(arr)
    for i in range(1, len(arr)):
        cur = arr[i]
        j = i - 1
        while j >= 0 and key(arr[j]) > key(cur):
            arr[j+1] = arr[j]
            j -= 1
        arr[j+1] = cur
    return arr


# 5. Shell Sort
def shell_sort(arr, key=lambda x: x):
    arr = list(arr)
    gap = len(arr) // 2
    while gap > 0:
        for i in range(gap, len(arr)):
            temp = arr[i]
            j = i
            while j >= gap and key(arr[j - gap]) > key(temp):
                arr[j] = arr[j - gap]
                j -= gap
            arr[j] = temp
        gap //= 2
    return arr


# 6. Radix Sort (by ticket number)
def radix_sort(arr, key=lambda x: x["number"]):
    if not arr:
        return arr
    max_val = max(key(x) for x in arr)
    exp = 1
    result = list(arr)
    while max_val // exp > 0:
        buckets = [[] for _ in range(10)]
        for item in result:
            buckets[(key(item) // exp) % 10].append(item)
        result = [item for b in buckets for item in b]
        exp *= 10
    return result


# 7. Bucket Sort (by wait_secs)
def bucket_sort(arr, key=lambda x: x["wait_secs"]):
    if not arr:
        return arr
    max_val = max(key(x) for x in arr) or 1
    n = len(arr)
    buckets = [[] for _ in range(n)]
    for item in arr:
        idx = min(int(key(item) / max_val * n), n - 1)
        buckets[idx].append(item)
    result = []
    for b in buckets:
        result.extend(insertion_sort(b, key))
    return result


# 8. Tim Sort (Python built-in)
def tim_sort(arr, key=lambda x: x):
    return sorted(arr, key=key)


SORT_MAP = {
    "1": ("Quick Sort",     lambda a: quick_sort(a,     key=lambda x: x["number"])),
    "2": ("Merge Sort",     lambda a: merge_sort(a,     key=lambda x: x["number"])),
    "3": ("Heap Sort",      lambda a: heap_sort(a,      key=lambda x: x["number"])),
    "4": ("Insertion Sort", lambda a: insertion_sort(a, key=lambda x: x["number"])),
    "5": ("Shell Sort",     lambda a: shell_sort(a,     key=lambda x: x["number"])),
    "6": ("Radix Sort",     lambda a: radix_sort(a)),
    "7": ("Bucket Sort",    lambda a: bucket_sort(a)),
    "8": ("Tim Sort",       lambda a: tim_sort(a,       key=lambda x: x["number"])),
}


# ──────────────────────────────────────────────
# BANK QUEUE SYSTEM (Main Engine)
# ──────────────────────────────────────────────
class BankQueueSystem:
    def __init__(self):
        self.waiting_queue = Queue()          # LinkedList Queue
        self.history_stack = Stack()          # Stack for history
        self.serving: dict | None = None
        self.all_tickets: list[dict] = []     # Array for search/sort
        self._ticket_counter = {}             # Instance-specific ticket counter

    # ── ENQUEUE ──────────────────────────────
    def issue_ticket(self, service_code: str) -> dict:
        if service_code not in SERVICES:
            raise ValueError("รหัสบริการไม่ถูกต้อง")
        ticket = new_ticket(service_code, self._ticket_counter) # Pass instance counter
        self.waiting_queue.enqueue(ticket)
        self.all_tickets.append(ticket)
        return ticket

    # ── CALL NEXT ────────────────────────────
    def call_next(self) -> dict | None:
        if self.serving:
            print("  ⚠️  ยังมีคิวที่กำลังให้บริการอยู่ กรุณาปิดคิวก่อน")
            return None

        if not self.waiting_queue.is_empty():
            ticket = self.waiting_queue.dequeue()
        else:
            return None
        ticket["status"]    = "SERVING"
        ticket["served_at"] = datetime.now().strftime("%H:%M:%S")
        self.serving        = ticket
        return ticket

    # ── CLOSE TICKET ─────────────────────────
    def close_ticket(self) -> dict | None:
        if not self.serving:
            return None
        self.serving["status"]   = "DONE"
        self.serving["done_at"]  = datetime.now().strftime("%H:%M:%S")
        self.history_stack.push(self.serving)
        closed = self.serving
        self.serving = None
        return closed

    # ── CANCEL ───────────────────────────────
    def cancel_ticket(self, ticket_id: str) -> bool:
        # Search in waiting queue
        tickets = self.waiting_queue.to_list()
        for t in tickets:
            if t["id"] == ticket_id:
                t["status"] = "CANCELLED"
                # rebuild queue without this ticket
                self.waiting_queue = Queue()
                for x in tickets:
                    if x["id"] != ticket_id:
                        self.waiting_queue.enqueue(x)
                self.history_stack.push(t)
                return True
        return False

    # ── SEARCH ───────────────────────────────
    def search(self, ticket_id: str) -> dict | None:
        # Linear search
        for t in self.all_tickets:
            if t["id"] == ticket_id:
                return t
        return None

    # ── SORT & DISPLAY ───────────────────────
    def sorted_tickets(self, algo_key="1") -> list:
        name, fn = SORT_MAP.get(algo_key, SORT_MAP["1"])
        return name, fn(list(self.all_tickets))

    # ── STATS ────────────────────────────────
    def stats(self):
        total   = len(self.all_tickets)
        waiting = sum(1 for t in self.all_tickets if t["status"] == "WAITING")
        done    = sum(1 for t in self.all_tickets if t["status"] == "DONE")
        cancelled = sum(1 for t in self.all_tickets if t["status"] == "CANCELLED")
        return {"total": total, "waiting": waiting, "done": done, "cancelled": cancelled}

    # ── SAVE/LOAD STATE ──────────────────────
    def save_state(self, filename="bank_queue_state.json"):
        state = {
            "waiting_queue": self.waiting_queue.to_list(),
            "history_stack": self.history_stack._data, # Access internal list for direct save
            "serving": self.serving,
            "all_tickets": self.all_tickets,
            "_ticket_counter": self._ticket_counter
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        print(f"  ✅ บันทึกสถานะไปที่ '{filename}' แล้ว")

    def load_state(self, filename="bank_queue_state.json"):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                state = json.load(f)

            self.waiting_queue = Queue()
            for item in state.get("waiting_queue", []):
                self.waiting_queue.enqueue(item)

            self.history_stack = Stack()
            self.history_stack.from_list(state.get("history_stack", [])) # Use from_list to restore

            self.serving = state.get("serving", None)
            self.all_tickets = state.get("all_tickets", [])
            self._ticket_counter = state.get("_ticket_counter", {})
            print(f"  ✅ โหลดสถานะจาก '{filename}' แล้ว")
            return True
        except FileNotFoundError:
            print(f"  ❌ ไม่พบไฟล์สถานะ '{filename}' เริ่มระบบใหม่")
            return False
        except json.JSONDecodeError:
            print(f"  ❌ เกิดข้อผิดพลาดในการอ่านไฟล์สถานะ '{filename}' เริ่มระบบใหม่")
            return False


# ──────────────────────────────────────────────
# UI HELPERS
# ──────────────────────────────────────────────
LINE = "─" * 52

def header(title: str):
    print(f"\n{'═'*52}")
    print(f"  🏦  {title}")
    print(f"{'═'*52}")

def section(title: str):
    print(f"\n  {LINE}")
    print(f"  {title}")
    print(f"  {LINE}")

def ticket_card(t: dict):
    print(f"""
  ┌─────────────────────────────────┐
  │  🎫  คิว: {t['id']:<6}  {STATUS[t['status']]}
  │  บริการ: {t['service']}
  │  ออกคิวเวลา: {t['issued_at']}
  └─────────────────────────────────┘""")


# ──────────────────────────────────────────────
# MENUS
# ──────────────────────────────────────────────
def menu_main():
    print(f"""
  {'─'*40}
  1) 🎫  ออกบัตรคิว
  2) 📢  เรียกคิวถัดไป
  3) ✅  ปิดคิว (เสร็จสิ้น)
  4) ❌  ยกเลิกคิว
  5) 🔍  ค้นหาคิว
  6) 📋  แสดงคิวทั้งหมด (เลือก Sort)
  7) 📊  สถิติ
  8) 🕑  ประวัติ (Stack)
  9) 💾  บันทึกสถานะ
  10) 📂 โหลดสถานะ
  0) 🚪  ออก
  {'─'*40}""")


def menu_service():
    print("\n  เลือกบริการ:")
    for k, (name, prefix) in SERVICES.items():
        print(f"  {k}) [{prefix}] {name}")


def menu_sort():
    print("\n  เลือก Algorithm:")
    for k, (name, _) in SORT_MAP.items():
        print(f"  {k}) {name}")


# ──────────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────────
def main():
    sys = BankQueueSystem()
    sys.load_state() # Attempt to load state at startup
    atexit.register(sys.save_state) # Register auto-save on exit
    header("ระบบบัตรคิวธนาคาร  Bank Queue System")

    while True:
        s = sys.stats()
        print(f"\n  [รอ:{s['waiting']}  บริการ:{'1' if sys.serving else '0'}  เสร็จ:{s['done']}  ยกเลิก:{s['cancelled']}]")
        menu_main()
        choice = input("  เลือก: ").strip()

        # 1 ── ออกบัตรคิว
        if choice == "1":
            menu_service()
            sc = input("  รหัสบริการ: ").strip().upper()
            if sc not in SERVICES:
                print("  ❌ รหัสไม่ถูกต้อง")
                continue
            t = sys.issue_ticket(sc)
            section("ออกบัตรคิวสำเร็จ")
            ticket_card(t)
            pos = sys.waiting_queue.size
            print(f"  คิวก่อนหน้า: {pos} คน")

        # 2 ── เรียกคิวถัดไป
        elif choice == "2":
            t = sys.call_next()
            if t:
                section("เรียกคิว")
                ticket_card(t)
            else:
                print("  ℹ️  ไม่มีคิวในระบบ หรือยังมีคิวที่ให้บริการอยู่")

        # 3 ── ปิดคิว
        elif choice == "3":
            t = sys.close_ticket()
            if t:
                section("ปิดคิว - เสร็จสิ้น")
                ticket_card(t)
            else:
                print("  ⚠️  ไม่มีคิวที่กำลังให้บริการ")

        # 4 ── ยกเลิกคิว
        elif choice == "4":
            tid = input("  กรอกหมายเลขคิว (เช่น A001): ").strip().upper()
            if sys.cancel_ticket(tid):
                print(f"  ✅ ยกเลิกคิว {tid} แล้ว")
            else:
                print("  ❌ ไม่พบคิวนี้ในรายการรอ")

        # 5 ── ค้นหา
        elif choice == "5":
            tid = input("  กรอกหมายเลขคิว: ").strip().upper()
            t = sys.search(tid)
            if t:
                section(f"พบคิว: {tid}")
                ticket_card(t)
            else:
                print("  ❌ ไม่พบข้อมูลคิวนี้")

        # 6 ── แสดงทั้งหมด + เลือก Sort
        elif choice == "6":
            if not sys.all_tickets:
                print("  ℹ️  ยังไม่มีคิวในระบบ")
                continue
            menu_sort()
            ak = input("  เลือก: ").strip()
            if ak not in SORT_MAP:
                ak = "1"
            name, sorted_list = sys.sorted_tickets(ak)
            section(f"คิวทั้งหมด [{name}]")
            print(f"  {'คิว':<8} {'บริการ':<20} {'สถานะ':<18} {'เวลา'}")
            print(f"  {'─'*8} {'─'*20} {'─'*18} {'─'*8}")
            for t in sorted_list:
                print(f"  {t['id']:<6} {t['service']:<20} {STATUS[t['status']]:<18} {t['issued_at']}")

        # 7 ── สถิติ
        elif choice == "7":
            s = sys.stats()
            section("สถิติระบบ")
            print(f"  คิวทั้งหมด   : {s['total']}")
            print(f"  รอให้บริการ  : {s['waiting']}")
            print(f"  เสร็จสิ้น    : {s['done']}")
            print(f"  ยกเลิก       : {s['cancelled']}")
            if sys.serving:
                print(f"\n  กำลังให้บริการ: {sys.serving['id']} - {sys.serving['service']}")

        # 8 ── ประวัติ (Stack)
        elif choice == "8":
            history = sys.history_stack.to_list()
            section("ประวัติ (Stack - ล่าสุดก่อน)")
            if not history:
                print("  ℹ️  ยังไม่มีประวัติ")
            for t in history[:10]:
                print(f"  {t['id']} | {t['service']} | {STATUS[t['status']]} | {t.get('done_at') or t['issued_at']}")

        # 9 ── บันทึกสถานะ
        elif choice == "9":
            sys.save_state()

        # 10 ── โหลดสถานะ
        elif choice == "10":
            sys.load_state()

        # 0 ── ออก
        elif choice == "0":
            print("\n  👋 ขอบคุณที่ใช้บริการ  ลาก่อนครับ/ค่ะ\n")
            break

        else:
            print("  ❌ กรุณาเลือกเมนู 0-10")


if __name__ == "__main__":
    main()


In [1]:
pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.3 MB/s eta 0:00:00


3. Login **ngrok** ที่ https://ngrok.com/ and copy your **authtoken**

In [2]:
!ngrok authtoken 36appsvdE3emdP67jo6mhpbyKLG_3hHbeikkq5iJsgQwBXxRp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
from pyngrok import ngrok

ngrok.kill()  # ปิด tunnel เก่า

public_url = ngrok.connect(8501)
print("🌍 Open your app here:", public_url)

🌍 Open your app here: NgrokTunnel: "https://stipitate-stainably-addisyn.ngrok-free.dev" -> "http://localhost:8501"


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime
from pyngrok import ngrok
import os

# ──────────────────────────────────────────────
# DATA STRUCTURES (Logic เดิมของน้อง)
# ──────────────────────────────────────────────
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class Queue:
    def __init__(self):
        self.head = self.tail = None
        self.size = 0
    def enqueue(self, data):
        node = Node(data)
        if self.tail: self.tail.next = node
        self.tail = node
        if not self.head: self.head = node
        self.size += 1
    def dequeue(self):
        if not self.head: return None
        data = self.head.data
        self.head = self.head.next
        if not self.head: self.tail = None
        self.size -= 1
        return data

# ──────────────────────────────────────────────
# CONFIG & INITIALIZATION
# ──────────────────────────────────────────────
SERVICES = {
    "A": ("ฝาก/ถอนเงิน", "A"),
    "B": ("เปิดบัญชีใหม่", "B"),
    "C": ("สินเชื่อ", "C"),
    "D": ("บัตรเครดิต", "D"),
    "E": ("แลกเปลี่ยนเงินตรา", "E"),
}

# ใช้ Session State เพื่อเก็บข้อมูลไม่ให้หายเวลาหน้าจอ Refresh
if 'all_tickets' not in st.session_state:
    st.session_state.all_tickets = []
# Change waiting_list to waiting_queues (a dict of lists, one for each service prefix)
if 'waiting_queues' not in st.session_state:
    st.session_state.waiting_queues = {prefix: [] for _, prefix in SERVICES.values()}
# Change serving to serving_channels (a dict, one entry per service prefix)
if 'serving_channels' not in st.session_state:
    st.session_state.serving_channels = {prefix: None for _, prefix in SERVICES.values()}
if 'counter' not in st.session_state:
    st.session_state.counter = {v[1]: 0 for v in SERVICES.values()}

# ──────────────────────────────────────────────
# APP UI - STREAMLIT
# ──────────────────────────────────────────────
st.set_page_config(page_title="Bank Queue Management", layout="wide")

st.title("🏦 ระบบจัดการคิวธนาคาร (Streamlit Version)")
st.markdown("---")

# Layout: แบ่งเป็นหลายส่วนสำหรับออกบัตรคิว ช่องบริการแต่ละประเภท และการแสดงผล
# Adjusted columns for 5 service channels and 1 display column (issue will be in sidebar)
col_service_A, col_service_B, col_service_C, col_service_D, col_service_E = st.columns([1, 1, 1, 1, 1]) # Removed col_display here
service_cols = {
    "A": col_service_A,
    "B": col_service_B,
    "C": col_service_C,
    "D": col_service_D,
    "E": col_service_E,
}

# --- ส่วนที่ 1: ออกบัตรคิว (Customer Side) - Moved to Sidebar ---
with st.sidebar:
    with st.container(border=True):
        st.markdown("### 🎫 ออกบัตรคิว")
        for code, (name, prefix) in SERVICES.items():
            if st.button(f"[{prefix}] {name}", use_container_width=True, key=f"issue_{prefix}"): # Add unique key
                st.session_state.counter[prefix] += 1
                new_id = f"{prefix}{st.session_state.counter[prefix]:03d}"
                ticket = {
                    "id": new_id,
                    "service": name,
                    "prefix": prefix, # Store prefix for easy access
                    "status": "⏳ รอเรียก",
                    "issued_at": datetime.now().strftime("%H:%M:%S")
                }
                st.session_state.all_tickets.append(ticket)
                st.session_state.waiting_queues[prefix].append(ticket) # Add to specific service queue
                st.success(f"ออกคิวสำเร็จ: {new_id}")

# --- ส่วนที่ 2: เรียกคิว (Staff Side) - Multiple Channels ---
# Iterate through each service to create a dedicated service channel
for service_code, (service_name, prefix) in SERVICES.items():
    with service_cols[prefix]: # Use the specific column for this service
        with st.container(border=True):
            st.markdown(f"### 📢 ช่อง {prefix} ({service_name})")
            serving_ticket = st.session_state.serving_channels.get(prefix)

            if serving_ticket:
                st.warning(f"กำลังบริการ: **{serving_ticket['id']}**")
                st.write(f"บริการ: {serving_ticket['service']}")
                if st.button(f"✅ เสร็จสิ้นรายการ (ช่อง {prefix})", type="primary", use_container_width=True, key=f"close_{prefix}"):
                    # Update status in the main all_tickets list
                    for t in st.session_state.all_tickets:
                        if t['id'] == serving_ticket['id']:
                            t['status'] = "🏁 เสร็จสิ้น"
                    st.session_state.serving_channels[prefix] = None
                    st.rerun()
            else:
                st.info("สถานะ: ว่าง")
                if st.button(f"🔔 เรียกคิวถัดไป (ช่อง {prefix})", use_container_width=True, key=f"call_next_{prefix}"):
                    if st.session_state.waiting_queues[prefix]: # Check specific service queue
                        next_t = st.session_state.waiting_queues[prefix].pop(0)
                        next_t['status'] = "✅ กำลังบริการ"
                        st.session_state.serving_channels[prefix] = next_t
                        st.rerun()
                    else:
                        st.error(f"ไม่มีคิวรอสำหรับช่อง {prefix}")

# --- ส่วนที่ 3: แสดงผลและค้นหา (Admin/Display Side) - Now below service channels ---
with st.container(border=True): # Placed outside of st.columns
    st.markdown("### 📋 สถานะคิวทั้งหมด")

    # ส่วนของ Sorting (เลือก Algorithm ได้เหมือนโค้ดเดิม)
    sort_algo = st.selectbox("เรียงลำดับด้วย:", ["ตามเวลาที่ออก", "ตามหมายเลขคิว (Quick Sort)", "ตามบริการ"])

    if st.session_state.all_tickets:
        df = pd.DataFrame(st.session_state.all_tickets)

        # Apply sorting based on selection
        if sort_algo == "ตามเวลาที่ออก":
            df = df.sort_values(by="issued_at", ascending=True)
        elif sort_algo == "ตามหมายเลขคิว (Quick Sort)": # Using pandas sort for this representation
            df = df.sort_values(by="id", ascending=True)
        elif sort_algo == "ตามบริการ":
            df = df.sort_values(by="service", ascending=True)

        # ค้นหาคิว (Linear Search)
        search_query = st.text_input("🔍 ค้นหาหมายเลขคิว (เช่น A001):").upper()
        if search_query:
            df = df[df['id'].str.contains(search_query)]

        st.table(df)
    else:
        st.write("ยังไม่มีข้อมูลคิวในวันนี้")

# ปุ่ม Reset (สำหรับทดสอบใหม่)
with st.container(border=True):
    if st.button("🗑️ Reset System"):
        st.session_state.clear()
        st.rerun()

Writing app.py


4. **รัน app Streamlit** ที่สร้างขึ้นมา
การบอกให้ Colab รันไฟล์ calculator.py ด้วย Streamlit บนพอร์ต 8501

In [5]:
!streamlit run app.py --server.port 8501 &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.188.248.2:8501

  Stopping...
